# More explaination on agent components

<img src="tool run time.jpg">

Context, State, Store, and Stream Writer are components of the overarching LangChain/LangGraph Agent framework, not part of the ToolRuntime object itself.

<img src="context_state_store_stream_writer.jpg">

<img src="context_vs_state_vs_store.jpg">

## 1. Context (runtime.context)
- Goal: To provide static configuration and identity details.

- Analogy: Your Driver's License. It's fixed, essential for the current activity (the run), and doesn't change during the journey.

- Key Difference from Store: The Context is configured at the start of the run and is expected to be read-only. It contains session identifiers and environmental details, not data that the agent is meant to update and save for next week.

## 2. State (runtime.state)
- Goal: To track the dynamic flow and short-term memory of the current task.

- Analogy: A Scratchpad. It's constantly being read from, erased, and written to as the agent works on a complex problem.

- Key Difference from Store: State data is primarily transient. While it might survive a single agent.invoke() call, it's generally tied to the active session and is the data you would lose if the system crashed. It's meant for coordination, not permanent storage.

## 3. Store (runtime.store)
- Goal: To provide persistent, external memory across sessions.

- Analogy: A Database or Filing Cabinet. It is the place to save data that must be recalled days or weeks later.

- Key Difference from Context: The Store is writable and designed for updates. An agent calls a tool to write user preferences to the Store, and later calls a tool to read those preferences back. The Store is the direct integration point for true <b>long-term memory</b> or the databases used for <b>RAG</b>.

# My Questions

4. What if I want to do this myself without using an Agent?

If you want to use the tool functions you wrote but skip the entire LangChain Agent/LLM decision process, you can absolutely call them directly—but you have to manually provide the ToolRuntime object.

This is what you would do manually:

1. Create the State Data: Define the dictionary that holds all the necessary data.

2. Create the ToolRuntime Object: Manually import and create an instance of ToolRuntime, passing your state data to it.

3. Call the Tool: Call your function as a standard Python function, passing the required arguments and your manual ToolRuntime object.

In [ ]:
# Example
# Let's say you want to manually call get_user_preference('theme'):

# 1. & 2. Manually prepare the State and ToolRuntime object
# You would need to import ToolRuntime from LangChain, but conceptually:
from langgraph.prebuilt import ToolRuntime # Actual implementation detail

my_state_data = {
    "user_preferences": {
        "theme": "dark_mode",
        "language": "English"
    }
}

# The framework does this automatically, but you do it manually here:
manual_runtime = ToolRuntime(state=my_state_data)

# 3. Call the function directly
result = get_user_preference(
    pref_name="theme",
    runtime=manual_runtime
)

print(result) # Output: dark_mode

<b>Conclusion</b>: The Agent's magic is that it automates steps 1 and 2 (deciding when to call the tool and creating/populating the runtime), allowing you to focus purely on the tool's logic.

# State

and who manage the state?

The LangChain Agent framework itself manages the state.

Specifically, the core object that persists and manages the state across steps in an agentic workflow is the State object, which is often implemented using components from the LangGraph library (a key component of the advanced LangChain ecosystem).

Here is a breakdown of who manages the state and how:

The Developer defines what information is necessary in the state, and the LangChain/LangGraph Framework is responsible for updating and passing that state.

1. The Developer's Role (Defining the State)
As the developer, you decide what needs to be tracked.

- You define a State schema (e.g., a dictionary or Pydantic class).

- You tell the framework which parts of the state should be updated by certain tools or nodes (functions).

For your examples:

- You defined that the state needs to track "messages" for summarize_conversation.

- You defined that the state needs to track "user_preferences" for get_user_preference.

2. The Agent Framework's Role (Updating and Passing the State)
The framework performs the actual management tasks:

- Initialization: When an agent workflow starts, the framework creates the initial State object.

- Updates: After any action occurs (a user message is received, an LLM responds, or a tool runs), the framework runs an internal function to update the State with the new information.

- Example: When a user sends a new prompt, the framework updates the State["messages"] list by appending the new user message.

- Injection: When a tool needs to run, the framework takes the current, updated State, wraps it inside a ToolRuntime object, and injects it into the tool function.

<img src="agent_state_analogy.jpg" alt="Step 1 is the analyze calling tool in this graph">

# Before Agent framework

Overall summary = add previous prompt, agent response into new prompt on and on and on

That's a crucial distinction. When you remove the Agent framework, the only place an LLM has for "memory" or "state" is its Context Window.


Since the LLM model itself is inherently stateless (it doesn't remember anything from the last request), managing state becomes a job for the developer who controls the input prompt.

### LLM State Management Without an Agent
The entire state management process is reduced to one core concept: Context Engineering.

You must constantly re-send the necessary past information with every new request.

1. The Core Mechanism: The Context Window
What it is: The Context Window (or context length) is the finite amount of text, measured in tokens, that the LLM can "see" and process at any given moment. This includes your system instructions, the conversation history, your current question, and the space for the model's new answer.

How it Manages State: <b>To remember anything, the developer must package all the relevant state data (like the past conversation) into the prompt and send it inside this window.</b>

Analogy: The Note on the Fridge.

An LLM is a person with short-term memory loss. They forget everything they just said the moment the conversation ends.

To remind them what they said, you must write the entire past conversation (the "state") on a note and hand it to them every time you ask a new question. This note is the Context Window.

### The Three State Management Strategies
Without an agent framework, you are forced to manually implement these strategies to fit the conversation history into the limited context window:

1. Simple Buffering (The Brute-Force Method)

- How it Works: You simply store the full text of every single user message and AI response in a simple list or array. For every new turn, you compile that entire list into the prompt and send it to the LLM.

- The Problem: This is the least efficient and most common early failure point. Since the list grows with every message, you quickly run out of tokens (hit the Context Window limit), forcing the LLM to forget the beginning of the conversation.

2. The Sliding Window (The Chopping Method)

- How it Works: To prevent the prompt from exceeding the limit, you set a maximum number of tokens or messages ($k$). When a new message comes in, you delete the oldest messages from the beginning of the history to make space for the new one. This keeps a "sliding window" of the most recent interaction available to the LLM.

- Analogy: A ticket machine that only prints the last 10 tickets. The 11th ticket pushes the 1st one out.

3. Summarization (The Compression Method)
- How it Works: Once the conversation reaches a certain length, you use the LLM itself to compress the old messages. You send the first 50 messages to the LLM with the instruction: "Please summarize this conversation history into a single, concise paragraph."

- The resulting summary (which is much shorter) then permanently replaces the original 50 messages in your state history. This frees up tokens while preserving the key information.

- Example: Your stored state might look like: [Summary: "User is a data scientist who wants to start a $50k business."], [AI: "Understood."], [User: "What's the best business for me?"], [AI: "..." (current response)]

# Context

Breakdown of Context Access
Let's break down the components of that line:

1. runtime: This is the ToolRuntime object, the container the agent framework automatically injects into your tool function.

2. .context: This is the specific property within the ToolRuntime that holds the static, global data related to the current session or user. This is distinct from the .state property we discussed earlier, which usually holds the dynamic, conversational history.

3. .user_id: This is the specific field, defined by the generic type UserContext in the function signature, that you are retrieving from the context object. This ID is then used to look up data in the USER_DATABASE.

<img src="context.jpg">

runtime.context is usualy immutable